# Detective R-GNN — Warm-Start Imitation Logger

Genera un dataset di coppie **(state, action)** per fare il warm-start (behavior cloning) della GNN dei detective, imitando l'euristica `detective_engine` corrente.

**Per ogni decisione di un detective** vengono loggati:
- belief state (199,) prima della mossa
- one-hot delle 5 posizioni detective + posizione del detective che agisce
- ticket residui (Mr.X 4 + detective acting 3)
- last ticket usato da Mr.X + ultima posizione spotted + turni dallo spot
- legal action mask (199, 3) [taxi, bus, underground]
- **target**: chosen_dest (1..199) + chosen_vehicle
- esito della partita (per analisi/curriculum learning)

Mr.X gioca con MCTS **indebolito** per evitare un dataset sbilanciato 99% loss-detective.

Output:
- `samples.parquet` (metadati per riga, ~kB/riga)
- `tensors.npz` (belief, legal_mask, det_one_hot — indicizzati per `step_idx`)
- `games.parquet` (un riga per partita: winner, lunghezza, ecc.)
- `dataset_meta.json` (config + stats)
- `scotland_yard_warmstart.zip` (tutto insieme, scaricabile)

## 1. Setup: clone repo + dipendenze

In [ ]:
%%bash
set -e
cd /content
if [ ! -d scotland_yard ]; then
  git clone --depth 1 https://github.com/Jacopo888/scotland_yard.git
fi
ls scotland_yard

In [ ]:
!pip install -q networkx numpy pandas pyarrow tqdm

## 2. Mount Google Drive (output persistente)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/scotland_yard_dataset'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive output dir:', DRIVE_DIR)

## 3. Imports e setup path

In [ ]:
import os, sys, json, time, random, pickle, zipfile
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REPO_DIR = '/content/scotland_yard'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Le matrici dentro il repo sono caricate con path relativo './Matrix_generation/...'
# quindi il cwd deve essere REPO_DIR (già settato sopra).
import game as game_mod
import detective_engine as de_mod
import mrx_engine as mrx_mod
from game import Game, ADJ, BOARD_GRAPH, SHORTEST_PATH_TENSOR, ticket_bitmask
from detective_engine import DetectiveEngine
from mrx_engine import MrxEngine

print('Repo OK. Nodes:', len(BOARD_GRAPH.nodes()))
print('Shortest path tensor shape:', SHORTEST_PATH_TENSOR.shape)

## 4. Configurazione run

In [ ]:
# ---- Dataset / generazione ----
NUM_GAMES = 500           # partite da loggare
RANDOM_SEED = 42

# ---- MCTS indebolito (per bilanciare win-rate) ----
# Originale: NUM_EXPLORATIONS=20, NUM_SIMULATIONS=35  -> Mr.X ~99% winrate
# Sweep su 15 partite (sandbox) ha dato:
#   explo=6  sims=5  -> 13% Mr.X wins  (sbilanciato verso detective)
#   explo=8  sims=8  -> 47% Mr.X wins  (bilanciato, ~0.44s/game)
#   explo=10 sims=12 -> 67% Mr.X wins  (sbilanciato verso Mr.X)
# Tieni 8/8 per dataset bilanciato. Alza/abbassa per spostare la curva.
WEAK_NUM_EXPLORATIONS = 8
WEAK_NUM_SIMULATIONS = 8

# ---- Output ----
OUT_DIR_LOCAL = '/content/scotland_yard_dataset'
os.makedirs(OUT_DIR_LOCAL, exist_ok=True)

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = os.path.join(OUT_DIR_LOCAL, f'run_{RUN_TAG}')
os.makedirs(RUN_DIR, exist_ok=True)
print('Local run dir:', RUN_DIR)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Patch costanti del modulo MCTS in-place
mrx_mod.NUM_EXPLORATIONS = WEAK_NUM_EXPLORATIONS
mrx_mod.NUM_SIMULATIONS = WEAK_NUM_SIMULATIONS
print(f'Patched MCTS -> NUM_EXPLORATIONS={mrx_mod.NUM_EXPLORATIONS}, '
      f'NUM_SIMULATIONS={mrx_mod.NUM_SIMULATIONS}')

## 5. Funzione di logging

Replichiamo la logica dell'euristica `Game.detective_automated_turn` (argmax su belief + shortest path) ma capturando lo stato **prima** della mossa, l'azione scelta e la legal mask. La mossa viene poi applicata identicamente al codice originale, quindi la dinamica della partita non cambia.

In [ ]:
VEHICLES = ('taxi', 'bus', 'underground')  # niente water per i detective
VEH2IDX = {v: i for i, v in enumerate(VEHICLES)}

N_NODES = 199


def _legal_action_mask(game, det_id):
    """Bool mask (199, 3) -> (dest_node_idx, vehicle_idx) legali per detective det_id."""
    mask = np.zeros((N_NODES, 3), dtype=bool)
    pos = game.detectives_pos[det_id]
    occupied = set(game.detectives_pos[:det_id] + game.detectives_pos[det_id + 1:])
    tickets = game.detective_tickets[det_id]
    for neighbor, types in ADJ[pos].items():
        if neighbor in occupied:
            continue
        for t in types:
            if t in VEH2IDX and tickets.get(t, 0) > 0:
                mask[int(neighbor) - 1, VEH2IDX[t]] = True
    return mask


def _detectives_one_hot(detectives_pos):
    """(5, 199) bool one-hot."""
    oh = np.zeros((len(detectives_pos), N_NODES), dtype=bool)
    for i, p in enumerate(detectives_pos):
        oh[i, int(p) - 1] = True
    return oh


def _logged_detective_turn(game, belief_state, det_id):
    """Esegue la stessa logica di Game.detective_automated_turn ma
    ritorna (chosen_dest_int, chosen_vehicle_str). Se nessuna mossa
    legale: ritorna (None, None) e il game non viene modificato."""
    pos = game.detectives_pos[det_id]
    occupied = (game.detectives_pos[:det_id]
                + game.detectives_pos[det_id + 1:])
    reachable = [n for n in ADJ[pos]
                 if n not in occupied and game.has_tickets(det_id, n)]
    if not reachable:
        return None, None

    target = belief_state.argmax()  # 0-indexed node
    matrix_idx = ticket_bitmask(game.detective_tickets[det_id])
    candidates = [(n, SHORTEST_PATH_TENSOR[matrix_idx][int(n) - 1][target])
                  for n in reachable]
    best_node, _ = min(candidates, key=lambda x: x[1])
    vehicle = game.use_ticket(det_id, pos, best_node)
    game.detectives_pos[det_id] = best_node
    return int(best_node), vehicle


def play_logged(game_id):
    """Gioca una partita e restituisce:
       rows: list[dict] (una entry per decisione detective)
       tensors: dict con liste di np.array allineate a rows
       game_meta: dict con info sulla partita
    """
    game = Game()
    engine = DetectiveEngine(game.detectives_pos)
    mrx = MrxEngine(game, engine)

    rows = []
    beliefs = []
    legal_masks = []
    det_one_hots = []

    last_mrx_ticket = None
    last_spotted_pos = 0  # 0 = mai spotted
    last_spot_turn = -999

    while True:
        if game.check_victory(silent=True):
            break

        game_over = False
        # ---- Detective turn (uno alla volta, log per ognuno) ----
        for det_id in range(game.num_detectives):
            # Snapshot stato PRE-mossa
            pos_before = int(game.detectives_pos[det_id])
            belief_snap = engine.belief_state.copy().astype(np.float32)
            mask = _legal_action_mask(game, det_id)
            det_oh = _detectives_one_hot(game.detectives_pos)
            mrx_t = game.mrx_tickets
            d_tk = game.detective_tickets[det_id]
            all_det_tk = game.detective_tickets  # list of 5 dicts at this exact moment

            chosen_dest, chosen_vehicle = _logged_detective_turn(
                game, engine.belief_state, det_id)

            if chosen_dest is None:
                # Detective bloccato: niente sample (nessuna azione da imitare)
                if game.check_victory(silent=True):
                    game_over = True
                    break
                continue

            rows.append({
                'game_id': game_id,
                'turn': int(game.turn),
                'detective_id': int(det_id),
                'pos_before': pos_before,
                'mrx_pos': int(game.mrx_pos),  # GT, NON usare come feature
                'detectives_pos': [int(p) for p in game.detectives_pos],
                'mrx_tk_taxi': mrx_t['taxi'],
                'mrx_tk_bus': mrx_t['bus'],
                'mrx_tk_underground': mrx_t['underground'],
                'mrx_tk_water': mrx_t['water'],
                'det_tk_taxi': d_tk['taxi'],
                'det_tk_bus': d_tk['bus'],
                'det_tk_underground': d_tk['underground'],
                # All 5 detectives' tickets at this decision moment (per Design Doc §4)
                'all_det_tk_taxi': [t['taxi'] for t in all_det_tk],
                'all_det_tk_bus': [t['bus'] for t in all_det_tk],
                'all_det_tk_underground': [t['underground'] for t in all_det_tk],
                'last_mrx_ticket': last_mrx_ticket if last_mrx_ticket else '',
                'last_spotted_pos': last_spotted_pos,
                'turns_since_spot': (int(game.turn) - last_spot_turn) if last_spotted_pos else -1,
                'turns_to_next_spot': (5 - (int(game.turn) % 5)) % 5,
                'chosen_dest': chosen_dest,             # TARGET
                'chosen_vehicle': chosen_vehicle,       # TARGET
                'chosen_vehicle_idx': VEH2IDX[chosen_vehicle],  # TARGET
            })
            beliefs.append(belief_snap)
            legal_masks.append(mask)
            det_one_hots.append(det_oh)

            if game.check_victory(silent=True):
                game_over = True
                break
            engine.kalman_filter()

        game.detectives_moves.append(game.detectives_pos[:])
        if game_over:
            break

        # ---- Mr.X turn (MCTS indebolito) ----
        best_pos, best_ticket = mrx.search()
        game.x_automated_turn(best_pos, best_ticket)
        if best_ticket is not None:
            last_mrx_ticket = best_ticket

        if game.check_victory(silent=True):
            break

        engine.update_belief_after_mrx_move(best_ticket)

        if game.turn % 5 == 0:
            engine.mrx_is_spotted(game.mrx_pos)
            last_spotted_pos = int(game.mrx_pos)
            last_spot_turn = int(game.turn)

    winner = game.winner  # 0 = detectives, 1 = Mr.X
    game_meta = {
        'game_id': game_id,
        'winner': int(winner),
        'mrx_won': bool(winner == 1),
        'final_turn': int(game.turn),
        'n_decisions': len(rows),
        'final_mrx_pos': int(game.mrx_pos),
        'final_detectives_pos': [int(p) for p in game.detectives_pos],
    }
    # Annoto winner anche su ogni riga (utile per filtering / curriculum)
    for r in rows:
        r['winner'] = int(winner)
        r['mrx_won'] = bool(winner == 1)
    tensors = {
        'beliefs': beliefs,
        'legal_masks': legal_masks,
        'det_one_hots': det_one_hots,
    }
    return rows, tensors, game_meta


# Smoke test su una partita
_t0 = time.time()
_rows, _tens, _meta = play_logged(game_id=-1)
print(f'smoke: {len(_rows)} decisioni in {time.time()-_t0:.2f}s, meta={_meta}')

## 6. Loop di generazione dataset

Tutti i samples accumulati in RAM (per 500 partite stimiamo ~50k samples × (199 + 199·5 + 199·3) ≈ ~1.7k float per sample ≈ ~340 MB di tensori in float32 / bool). Compatibile con i 12GB di Colab.

In [ ]:
all_rows = []
all_beliefs = []
all_masks = []
all_det_oh = []
games_meta = []

t_start = time.time()
for gid in tqdm(range(NUM_GAMES), desc='games'):
    rows, tens, meta = play_logged(game_id=gid)
    all_rows.extend(rows)
    all_beliefs.extend(tens['beliefs'])
    all_masks.extend(tens['legal_masks'])
    all_det_oh.extend(tens['det_one_hots'])
    games_meta.append(meta)
elapsed = time.time() - t_start
print(f'Done. {len(all_rows)} samples da {NUM_GAMES} partite in {elapsed/60:.1f} min')
print(f'Avg decisioni/partita: {len(all_rows)/NUM_GAMES:.1f}')

## 7. Serializzazione: Parquet + NPZ

In [ ]:
# step_idx: chiave di join fra parquet e npz
for i, r in enumerate(all_rows):
    r['step_idx'] = i

df = pd.DataFrame(all_rows)
games_df = pd.DataFrame(games_meta)

samples_path = os.path.join(RUN_DIR, 'samples.parquet')
games_path = os.path.join(RUN_DIR, 'games.parquet')
tensors_path = os.path.join(RUN_DIR, 'tensors.npz')
meta_path = os.path.join(RUN_DIR, 'dataset_meta.json')

df.to_parquet(samples_path, index=False)
games_df.to_parquet(games_path, index=False)

# Tensori stacked
beliefs_arr = np.stack(all_beliefs).astype(np.float32)              # (N, 199)
masks_arr = np.stack(all_masks).astype(np.bool_)                    # (N, 199, 3)
det_oh_arr = np.stack(all_det_oh).astype(np.bool_)                  # (N, 5, 199)

np.savez_compressed(
    tensors_path,
    beliefs=beliefs_arr,
    legal_masks=masks_arr,
    detectives_one_hot=det_oh_arr,
)

meta = {
    'run_tag': RUN_TAG,
    'num_games': NUM_GAMES,
    'random_seed': RANDOM_SEED,
    'weak_num_explorations': WEAK_NUM_EXPLORATIONS,
    'weak_num_simulations': WEAK_NUM_SIMULATIONS,
    'n_samples': int(len(df)),
    'shapes': {
        'beliefs': list(beliefs_arr.shape),
        'legal_masks': list(masks_arr.shape),
        'detectives_one_hot': list(det_oh_arr.shape),
    },
    'columns': list(df.columns),
    'vehicles': list(VEHICLES),
    'n_nodes': N_NODES,
    'mrx_winrate': float(games_df['mrx_won'].mean()),
    'avg_game_length_turns': float(games_df['final_turn'].mean()),
    'elapsed_seconds': elapsed,
    'generated_at': datetime.now().isoformat(),
    'repo': 'https://github.com/Jacopo888/scotland_yard',
}
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

for p in (samples_path, games_path, tensors_path, meta_path):
    sz = os.path.getsize(p) / (1024 * 1024)
    print(f'{p}  {sz:.2f} MB')

## 8. Sanity check + statistiche

In [ ]:
print('=== Win rate ===')
print(f'Mr.X wins: {int(games_df["mrx_won"].sum())}/{len(games_df)} '
      f'({games_df["mrx_won"].mean()*100:.1f}%)')
print(f'Detectives wins: {int((~games_df["mrx_won"]).sum())}/{len(games_df)} '
      f'({(~games_df["mrx_won"]).mean()*100:.1f}%)')

print('\n=== Lunghezza partita ===')
print(games_df['final_turn'].describe())

print('\n=== Distribuzione vehicle scelto ===')
print(df['chosen_vehicle'].value_counts())

print('\n=== Decisioni per turno ===')
print(df.groupby('turn').size().describe())

print('\n=== Decisioni per detective_id ===')
print(df['detective_id'].value_counts().sort_index())

print('\n=== Sanity tensori ===')
print('beliefs sum~1:', np.allclose(beliefs_arr.sum(axis=1), 1.0, atol=1e-3).mean())
print('legal_masks: almeno 1 mossa legale per riga ->',
      (masks_arr.reshape(len(masks_arr), -1).sum(axis=1) > 0).mean())
print('det_one_hot: 5 detective per sample ->',
      (det_oh_arr.sum(axis=(1, 2)) == 5).mean())

# Verifica che la chosen action sia sempre in legal_mask
_legal_chosen = 0
for i, r in df.iterrows():
    if masks_arr[r['step_idx'], r['chosen_dest'] - 1, r['chosen_vehicle_idx']]:
        _legal_chosen += 1
print(f'chosen action ∈ legal_mask: {_legal_chosen}/{len(df)} '
      f'({_legal_chosen/len(df)*100:.2f}%)')

## 9. Copia su Drive + zip per download

In [ ]:
import shutil

drive_run_dir = os.path.join(DRIVE_DIR, f'run_{RUN_TAG}')
os.makedirs(drive_run_dir, exist_ok=True)
for p in (samples_path, games_path, tensors_path, meta_path):
    shutil.copy2(p, drive_run_dir)
print('Copiato in Drive:', drive_run_dir)

# Zip scaricabile
zip_path = f'/content/scotland_yard_warmstart_{RUN_TAG}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in (samples_path, games_path, tensors_path, meta_path):
        zf.write(p, arcname=os.path.basename(p))
print('Zip pronto:', zip_path, f'({os.path.getsize(zip_path)/(1024*1024):.2f} MB)')

# Download diretto
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('Download manuale dalla sidebar di Colab. Errore auto-download:', e)

## 10. (Opzionale) Quick load per verifica

Snippet che rifletteresti nel training script per caricare il dataset:

In [ ]:
df_loaded = pd.read_parquet(samples_path)
tens_loaded = np.load(tensors_path)
print('df shape:', df_loaded.shape)
print('tensors:', {k: tens_loaded[k].shape for k in tens_loaded.files})

# Esempio: un sample
row = df_loaded.iloc[0]
i = row['step_idx']
print('\n--- sample 0 ---')
print('belief argmax node:', int(tens_loaded['beliefs'][i].argmax()) + 1,
      'p=', float(tens_loaded['beliefs'][i].max()))
print('legal moves:', int(tens_loaded['legal_masks'][i].sum()))
print('chosen:', row['chosen_dest'], row['chosen_vehicle'])
print('detective acting:', row['detective_id'], 'pos:', row['pos_before'])
print('winner:', row['winner'], '(0=det, 1=mrx)')